# SWDL Resumable Training on Google Colab (BHSD Dataset)

Notebook này được thiết kế để huấn luyện mô hình SWDL với bộ dữ liệu BHSD trên Google Colab. 
Để đối phó với giới hạn thời gian phiên của Colab (có thể bị ngắt kết nối giữa chừng), hệ thống huấn luyện đã được tích hợp tính năng **tự động lưu checkpoint và khôi phục (Resume)** từ Google Drive.

### Quy trình chuẩn bị trước khi chạy:
1. Tải toàn bộ thư mục dự án `SWDL` của bạn lên Google Drive tại thư mục gốc (`/content/drive/MyDrive/SWDL`).
2. Nén thư mục dataset `BHSD_Dataset_RemoveSkull_resampled` thành file `.zip` (ví dụ: `BHSD_Dataset_RemoveSkull_resampled.zip`) và đặt nó tại `/content/drive/MyDrive/SWDL/data/BHSD_Dataset_RemoveSkull_resampled.zip`.
3. Mở notebook này trên Google Colab trực tiếp từ Google Drive của bạn.

In [ ]:
# Bước 1: Kết nối Google Drive để truy cập mã nguồn và lưu checkpoint
from google.colab import drive
drive.mount('/content/drive')

## Bước 2: Cài đặt các thư viện cần thiết
Cài đặt các thư viện y tế và ghi log đặc thù (`SimpleITK`, `medpy`, `tensorboardX`, `nibabel`) vốn không có sẵn trên Colab.

In [ ]:
!pip install SimpleITK medpy tensorboardX nibabel

## Bước 3: Chuẩn bị dữ liệu cục bộ
Đọc dữ liệu trực tiếp từ Google Drive có thể rất chậm do độ trễ kết nối. Vì vậy, ta sẽ copy file nén từ Drive sang ổ cứng SSD cục bộ của Colab (`/content/`) và giải nén tại đó để huấn luyện với tốc độ tối đa.

In [ ]:
import os

# Đường dẫn tới file zip dataset trên Google Drive của bạn
drive_zip_path = "/content/drive/MyDrive/SWDL/data/BHSD_Dataset_RemoveSkull_resampled.zip"

# Thư mục cục bộ dùng để giải nén
local_dataset_dir = "/content/BHSD_Dataset_RemoveSkull_resampled"

if not os.path.exists(local_dataset_dir):
    if os.path.exists(drive_zip_path):
        print(f"Đang sao chép và giải nén {drive_zip_path} sang SSD cục bộ...")
        os.makedirs(local_dataset_dir, exist_ok=True)
        # Giải nén trực tiếp vào /content/
        !unzip -q {drive_zip_path} -d /content/
        print("Giải nén thành công!")
    else:
        print(f"LỖI: Không tìm thấy file dataset zip tại: {drive_zip_path}")
        print("Vui lòng kiểm tra lại đường dẫn trên Google Drive của bạn.")
else:
    print("Dataset đã được giải nén sẵn trên SSD cục bộ.")

## Bước 4: Tiến hành Huấn luyện & Khôi phục (Resume)

Chạy lệnh huấn luyện với tùy chọn `--resume`. 
- Nếu đây là lần chạy đầu tiên, mô hình sẽ huấn luyện từ đầu.
- Nếu phiên bị ngắt kết nối giữa chừng, khi bạn chạy lại cell này, mã nguồn sẽ tự động tải checkpoint mới nhất (`latest_checkpoint.pth`) được lưu trữ an toàn trên Google Drive của bạn, bao gồm:
  - Trọng số mô hình (Model Weights)
  - Trạng thái bộ tối ưu hóa (Optimizer States - learning rate, momentum...)
  - Số lượng iteration hiện tại (Iteration Count)
  - Trạng thái bộ sinh số ngẫu nhiên (Random Generator Seeds) để đảm bảo tính nhất quán.

Mọi checkpoint và file logs sẽ được lưu trực tiếp vào thư mục `SWDL/model/` trên Google Drive của bạn nên sẽ không bao giờ bị mất.

In [ ]:
# Di chuyển thư mục làm việc đến thư mục code trên Drive
%cd /content/drive/MyDrive/SWDL/code/BHSD2024

# Chạy script huấn luyện với đường dẫn dữ liệu cục bộ và bật cờ --resume
!python train_SWDL.py --root_path /content/BHSD_Dataset_RemoveSkull_resampled/dataSet --labeled_num 153 --resume

## Bước 5: Theo dõi tiến trình bằng TensorBoard
Nhờ logs được lưu trực tiếp trên Google Drive, bạn có thể theo dõi loss, dice score trực tiếp ngay cả khi đang huấn luyện hoặc xem lại sau khi kết thúc.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/SWDL/model/BHSD/